# We will be formatting the data for Muktar's paper

# *NOTE: WE WILL NEED TO GIVE HIM THE MEDIANS OF THE CELL DATA*

In [2]:
import polars as pl
from pathlib import Path
import shutil
import os
from time import sleep
from subprocess import run

feb_data_pheno = Path("/home/derfelt/PartchLabFiles/Circadian Rhythms/Live "
                "Study/analysis_ready_data/2026-03-11CelPaint_PhenovN"
                "oTreatDex_HARMONY_FIXED._.TIME-CORRECTED.tsv")

feb_data_chroma = Path("/home/derfelt/PartchLabFiles/Circadian Rhythms/Live "
                       "Study/analysis_ready_data/2026-03-10CelPaint_ChromaN"
                       "oTreatDex._.TIME-CORRECTED.tsv")

out_dir = Path("/home/derfelt/PartchLabFiles/Circadian Rhythms/Live "
               "Study/SONAR_data_package")

def clean_dir(path: Path, delete_folder: bool = False):
    if os.path.exists(path):
        shutil.rmtree(path)
        sleep(0.5)
    
    if not delete_folder:
        path.mkdir(parents=True, exist_ok=True)

# lf = pl.scan_csv(feb_data_pheno, separator="\t").schema.to_python()

# print(*list(lf.keys()), sep="\n")

In [3]:
parquet_out = out_dir / f"{feb_data_chroma.stem}.parquet"

run([
    "ccellpaint", "preprocess",
    feb_data_chroma, parquet_out,
    "-w", "WellName",
    "-t", "Timepoint",
    "-f",
])

INFO:root:4755 features -> 20 batches of 250
Batching progress: 100%|██████████| 20/20 [37:30<00:00, 112.53s/it]


CompletedProcess(args=['ccellpaint', 'preprocess', PosixPath('/home/derfelt/PartchLabFiles/Circadian Rhythms/Live Study/analysis_ready_data/2026-03-10CelPaint_ChromaNoTreatDex._.TIME-CORRECTED.tsv'), PosixPath('/home/derfelt/PartchLabFiles/Circadian Rhythms/Live Study/SONAR_data_package/2026-03-10CelPaint_ChromaNoTreatDex._.TIME-CORRECTED.parquet'), '-w', 'WellName', '-t', 'Timepoint', '-f'], returncode=0)

In [5]:
feb_preprocessed = Path("/home/derfelt/PartchLabFiles/Circadian Rhythms/Live "
                        "Study/SONAR_data_package/2026-03-10CelPaint_ChromaNo"
                        "TreatDex._.TIME-CORRECTED.parquet")

lf = pl.scan_parquet(feb_preprocessed)

display(lf.head(10).collect())
print(*list(lf.schema.to_python().keys()), sep="\n")

WellName,Timepoint,feature,mean,median,stddev,q0,q10,q25,q50,q75,q90,q100
str,f64,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""G12""",18.0,"""NoBorderObj - Cell Region Area…",1032.550555,940.412,531.439171,157.203,478.2064,680.044,940.412,1284.64,1654.216,5433.69
"""G12""",18.0,"""NoBorderObj - Cell Region Chro…",0.658881,0.666678,0.14993,0.256686,0.4561152,0.550886,0.666678,0.776343,0.8573642,0.988293
"""G12""",18.0,"""NoBorderObj - Cell Region Chro…",0.647598,0.644855,0.15367,0.253239,0.43493,0.533309,0.644855,0.76449,0.8586788,0.985934
"""G12""",18.0,"""NoBorderObj - Cell Region Chro…",0.659045,0.659818,0.150036,0.266293,0.4556364,0.549647,0.659818,0.777041,0.8579022,0.988116
"""G12""",18.0,"""NoBorderObj - Cell Region Chro…",0.662711,0.667543,0.150844,0.268076,0.4588804,0.54879,0.667543,0.779421,0.8686886,0.991129
"""G12""",18.0,"""NoBorderObj - Cell Region Chro…",0.652058,0.654029,0.150066,0.243055,0.453539,0.543765,0.654029,0.763839,0.8554332,0.981957
"""G12""",18.0,"""NoBorderObj - Cell Region Chro…",0.651182,0.65056,0.15264,0.254715,0.4417778,0.539214,0.65056,0.766508,0.858593,0.981607
"""G12""",18.0,"""NoBorderObj - Cell Region Chro…",0.653327,0.649876,0.152435,0.261427,0.4527454,0.541594,0.649876,0.767184,0.8573396,0.982104
"""G12""",18.0,"""NoBorderObj - Cell Region Chro…",0.638388,0.639413,0.15774,0.248514,0.424297,0.519444,0.639413,0.762566,0.848279,0.983943


WellName
Timepoint
feature
mean
median
stddev
q0
q10
q25
q50
q75
q90
q100


/tmp/ipykernel_65737/3278704284.py:8: PerformanceWarning: Resolving the schema of a LazyFrame is a potentially expensive operation. Use `LazyFrame.collect_schema()` to get the schema without this warning.
  print(*list(lf.schema.to_python().keys()), sep="\n")


## Let's write a loop that will create a temp `.parquet` file

In [ ]:
# parquet_out = out_dir / f"{feb_data_pheno.stem}.parquet"
# pl.scan_csv(feb_data, separator="\t").sink_parquet(parquet_out)
# feb_preprocessed = Path("/home/derfelt/PartchLabFiles/Circadian Rhythms/Live "
#                         "Study/SONAR_data_package/2026-03-10CelPaint_ChromaNo"
#                         "TreatDex._.TIME-CORRECTED.parquet")

feb_preprocessed = Path("/home/derfelt/PartchLabFiles/Circadian Rhythms/Live "
                        "Study/SONAR_data_package/2026-03-10CelPaint_ChromaNo"
                        "TreatDex._.TIME-CORRECTED.txt")

out_path = out_dir / f"{feb_preprocessed.stem}"
clean_dir(out_path)

id_cols = ["WellName", "Timepoint"]

lf = pl.scan_parquet(f"{out_path}.parquet")
feat_cols = [c for c in lf.collect_schema().names() if c not in id_cols]
wells = lf.select("WellName").unique().collect()["WellName"].to_list()
timepoints = lf.select("Timepoint").unique().collect()["Timepoint"].sort().to_list()
tp_cols = [str(t) for t in timepoints]

(pl
 .scan_parquet(feb_preprocessed)
 .select(["WellName", "Timepoint", "feature", "median"])
 .pivot(
            "Timepoint",
            on_columns=timepoints,
            index=["WellName", "feature"],
            values="median"
        )
 .sink_csv(out_dir / f"{feb_preprocessed.stem}.txt", separator="\t")
)

for well in wells:
    well_long = (
        lf
        .filter(pl.col("WellName") == well)
        .select(["WellName", "Timepoint", "feature", "median"])
        .pivot(
            "Timepoint",
            on_columns=timepoints,
            index=["WellName", "feature"],
            values="median"
        )
        .drop("WellName")
    )
    well_long.sink_csv(out_path / f"{well}_medians.txt", separator="\t")
    

I need to drop the well column in the individual files

In [12]:
txt_data = Path("/home/derfelt/PartchLabFiles/Circadian Rhythms/Live "
                "Study/SONAR_data_package/2026-03-11CelPaint_PhenovNo"
                "TreatDex_HARMONY_FIXED._.TIME-CORRECTED.txt")

out_path = out_dir / txt_data.stem
clean_dir(out_path)

well_col = "WellName"

lf = pl.scan_csv(txt_data, separator="\t")
wells = lf.select("WellName").unique().collect()["WellName"].to_list()

for well in wells:
    well_lf = lf.filter(pl.col(well_col) == well).drop(well_col)
    well_lf.sink_csv(out_path / f"{well}_median.txt", separator="\t")
    

# A new SONAR data formatter has been implemented in cytoclock!

Let's try it out!

In [1]:
import polars as pl
from pathlib import Path
import shutil
import os
from time import sleep
from subprocess import run
from cytoclock.formatters.sonar_converter import format_sonar
from cytoclock.utils import clean_folder

In [3]:
out_dir = Path("/home/derfelt/PartchLabFiles/Circadian Rhythms/Live "
               "Study/SONAR_data_package_JUNEexp")

june_pheno_data = Path("/home/derfelt/PartchLabFiles/Circadian Rhythms/Live "
                       "Study/analysis_ready_data/2026-06-01_20xRep_PHENOVUE.tsv")

print(june_pheno_data.stem)

# dump_path = out_dir / ".temp_batches"
# clean_folder(dump_path, delete_only=False)

# print(dump_path)

# format_sonar(
#     input_data=june_pheno_data, 
#     out_file=out_dir / f"{june_pheno_data.stem}.txt", 
#     dump_path=dump_path,
#     metric="median",
#     well_col="WellName",
#     timepoint_col="Timepoint",
#     additional_metacol=None
# )

# clean_folder(dump_path, delete_only=True)

2026-06-01_20xRep_PHENOVUE
